<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-10/AI_Modul_10.1_Praktikum_Konsep_Computer_Vision.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 10.1: Praktikum Konsep dan Fundamental Computer Vision
**Mata Kuliah:** Visi Komputer dan Kecerdasan Buatan Terapan  
**Institut Pertanian Stiper (INSTIPER) Yogyakarta**  

---

### Capaian Pembelajaran Praktikum:
1. Mampu mensimulasikan model fisika pembentukan citra digital ($I = L \cdot R$) pada domain kanopi perkebunan kelapa sawit.
2. Mampu merumuskan dan mengimplementasikan proyeksi geometris kamera lubang jarum (*pinhole camera*) dari koordinat 3D dunia ke bidang piksel 2D.
3. Mampu mendeteksi, memodelkan, dan mengompensasi distorsi lensa radial (*barrel distortion*) menggunakan OpenCV.
4. Mampu menghitung metrik resolusi spasial *Ground Sampling Distance* (GSD) pada berbagai ketinggian terbang wahana drone pemantau kebun.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Konfigurasi visualisasi
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print(f'OpenCV Version : {cv2.__version__}')
print(f'NumPy Version  : {np.__version__}')


## 1. Simulasi Model Pembentukan Citra: I(x, y) = L(x, y) * R(x, y)
Intensitas citra digital yang ditangkap oleh sensor merupakan hasil perkalian antara **Iluminasi Lapangan $L(x, y)$** (cahaya datang dari matahari) dan **Reflektansi Fisik $R(x, y)$** (karakteristik biologis permukaan daun/kanopi sawit).

In [ ]:
width, height = 400, 300
Y, X = np.ogrid[:height, :width]
cx, cy = width // 2, height // 2

# 1. Komponen Reflektansi Fisik R(x, y): Pola Radial Tajuk Kelapa Sawit [0.1 - 0.9]
radius = np.sqrt((X - cx)**2 + (Y - cy)**2)
angle = np.arctan2(Y - cy, X - cx)
reflectance = 0.5 + 0.35 * np.sin(12 * angle) * np.exp(-radius / 120.0)
reflectance = np.clip(reflectance, 0.1, 0.9)

# 2. Komponen Iluminasi L(x, y): Gradien Cahaya Matahari Condong dari Kiri Atas
illumination = 0.4 + 0.5 * (1.0 - (X + Y) / (width + height))

# 3. Intensitas Citra Terukur I = L * R
image_intensity = illumination * reflectance
image_uint8 = np.clip(image_intensity * 255.0, 0, 255).astype(np.uint8)

# Visualisasi Dekomposisi 3 Komponen
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(illumination, cmap='magma')
axes[0].set_title('(A) Iluminasi Matahari L(x, y)')
axes[0].axis('off')

axes[1].imshow(reflectance, cmap='YlGn')
axes[1].set_title('(B) Reflektansi Daun R(x, y)')
axes[1].axis('off')

axes[2].imshow(image_uint8, cmap='gray')
axes[2].set_title('(C) Citra Terukur I = L * R')
axes[2].axis('off')
plt.tight_layout()
plt.show()


## 2. Geometri Kamera Lubang Jarum & Proyeksi Titik 3D Dunia ke 2D Piksel
Kita mengimplementasikan transformasi koordinat dunia $\mathbf{P}_w = [X_w, Y_w, Z_w]^T$ menuju ruang piksel citra $(u, v)$ menggunakan matriks intrinsik $\mathbf{K}$ dan matriks ekstrinsik $[\mathbf{R} \mid \mathbf{t}]$.

In [ ]:
# Parameter Intrinsik Kamera Drone
f_mm = 4.5           # Jarak fokal optik dalam mm
pixel_size_um = 3.0  # Ukuran piksel 3 mikrometer (0.003 mm)
fx = f_mm / (pixel_size_um * 1e-3)  # fx dalam piksel = 1500 px
fy = fx
cx_opt, cy_opt = 1920.0 / 2.0, 1080.0 / 2.0  # Resolusi Full HD

K = np.array([
    [fx, 0.0, cx_opt],
    [0.0, fy, cy_opt],
    [0.0, 0.0, 1.0]
], dtype=np.float64)

# Parameter Ekstrinsik Drone Terbang Nadir pada Ketinggian H = 40 meter
H_flight = 40.0
R_nadir = np.array([
    [1.0, 0.0, 0.0],
    [0.0, -1.0, 0.0],
    [0.0, 0.0, -1.0]
], dtype=np.float64)
t_drone = np.array([[0.0], [0.0], [H_flight]], dtype=np.float64)

# Koordinat 4 Pohon Sawit di Lapangan 3D (Z = 0 permukaan tanah)
palms_3d = np.array([
    [0.0, 0.0, 0.0],    # Pohon Pusat (Nadir)
    [8.0, 0.0, 0.0],    # Pohon Timur (+8m)
    [-8.0, 0.0, 0.0],   # Pohon Barat (-8m)
    [0.0, 8.0, 0.0]     # Pohon Utara (+8m)
])

pixels_2d = []
for P_world in palms_3d:
    # Transformasi Ekstrinsik
    P_cam = np.dot(R_nadir, P_world.reshape(3, 1)) + t_drone
    Xc, Yc, Zc = P_cam.flatten()
    # Transformasi Intrinsik Homogen
    u = fx * (Xc / Zc) + cx_opt
    v = fy * (Yc / Zc) + cy_opt
    pixels_2d.append((u, v))

print('=== HASIL PROYEKSI PINHOLE POHON SAWIT KE CITRA DRONE ===')
labels = ['Pusat (0,0)', 'Timur (+8,0)', 'Barat (-8,0)', 'Utara (0,+8)']
for label, (u, v) in zip(labels, pixels_2d):
    print(f'{label:<15}: u = {u:7.2f} px, v = {v:7.2f} px')


## 3. Pemodelan & Kompensasi Distorsi Radial Lensa (Barrel Distortion)
Kamera bersudut lebar (*wide-angle*) pada drone kerap menghasilkan distorsi lengkung barel ($k_1 < 0$). Kita menerapkan model distorsi Brown-Conrady dan mengoreksinya menggunakan `cv2.undistort`.

In [ ]:
# Sintesis Pola Kisi Kebun Beraturan (Grid of Trees)
grid_img = np.ones((600, 600), dtype=np.uint8) * 240
for i in range(30, 600, 40):
    cv2.line(grid_img, (i, 0), (i, 600), color=100, thickness=2)
    cv2.line(grid_img, (0, i), (600, i), color=100, thickness=2)

# Matriks Intrinsik Kamera Uji
K_test = np.array([
    [350.0, 0.0, 300.0],
    [0.0, 350.0, 300.0],
    [0.0, 0.0, 1.0]
], dtype=np.float64)

# Koefisien Distorsi Barel Kuat
dist_test = np.array([-0.35, 0.10, 0.0, 0.0, 0.0], dtype=np.float64)

# Simulasi Distorsi Barel melalui Peta Remap Terbalik
mapx, mapy = cv2.initUndistortRectifyMap(
    K_test, dist_test, None, K_test, (600, 600), cv2.CV_32FC1
)
distorted_grid = cv2.remap(grid_img, mapx, mapy, cv2.INTER_LINEAR)

# Pemulihan Geometri Menggunakan cv2.undistort
recovered_grid = cv2.undistort(distorted_grid, K_test, -dist_test)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(grid_img, cmap='gray')
axes[0].set_title('(A) Kisi Kebun Asli (Tanpa Distorsi)')
axes[0].axis('off')

axes[1].imshow(distorted_grid, cmap='gray')
axes[1].set_title('(B) Terdistorsi Barel Lensa Drone (k1 < 0)')
axes[1].axis('off')

axes[2].imshow(recovered_grid, cmap='gray')
axes[2].set_title('(C) Hasil Koreksi Geometri (Undistorted)')
axes[2].axis('off')
plt.tight_layout()
plt.show()


## 4. Analisis Resolusi Spasial: Ground Sampling Distance (GSD)
Ground Sampling Distance (GSD) merepresentasikan ukuran fisik di atas tanah yang dicakup oleh satu piksel tunggal kamera drone:
$$\text{GSD} = \frac{H \cdot d_x}{f}$$
Di mana $H$ adalah ketinggian terbang, $d_x$ adalah dimensi piksel sensor, dan $f$ adalah jarak fokal fisik lensa.

In [ ]:
altitudes = np.array([10.0, 20.0, 30.0, 50.0, 80.0, 100.0, 150.0])  # Ketinggian (meter)
focal_len = 0.0045   # 4.5 mm = 0.0045 meter
pixel_pitch = 3.0e-6 # 3.0 mikrometer = 3e-6 meter

# Perhitungan GSD dalam cm/piksel
gsd_cm = (altitudes * pixel_pitch / focal_len) * 100.0

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(altitudes, gsd_cm, marker='o', color='darkgreen', lw=2, label='Kurva GSD vs Altitude')
ax.axhline(5.0, color='red', linestyle='--', label='Batas Kritis Deteksi Pelepah Sawit (5 cm/px)')
ax.set_title('Karakteristik Ground Sampling Distance (GSD) Kamera Drone', fontsize=12, fontweight='bold')
ax.set_xlabel('Ketinggian Terbang Drone H (meter)', fontsize=10)
ax.set_ylabel('GSD Lapangan (cm / piksel)', fontsize=10)
ax.legend(fontsize=9)
ax.grid(True, linestyle=':', alpha=0.7)
plt.tight_layout()
plt.show()

print('=== TABEL PERHITUNGAN GSD PERKEBUNAN ===')
for alt, g in zip(altitudes, gsd_cm):
    status = 'Optimal Deteksi Pelepah' if g <= 5.0 else 'Hanya untuk Sensus Pohon'
    print(f'Altitude: {alt:5.1f} m | GSD: {g:6.2f} cm/px | Rekomendasi: {status}')
